# 第 5 章 · Tool Use（Google Search / ADK）

`google_search` 是**内建工具**：搜索在 Google 侧执行，`tools=[]` 里放的是框架导出的工具对象，
不是你写的函数。

所以「模型到底调没调工具」代码里看不出来——只能查 `grounding_metadata`。
这也正是本章那条规律：**工具越内建，可观测性越差**。

kernel `Python (agentic-adk)`。


In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, short_error, retry_call
from google.adk.agents import LlmAgent          # 【适配】书里 import Agent 却用 ADKAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

USER_ID = "user1234"

# ★ 本章新增：内建工具。不是本地函数，搜索发生在 Google 侧。
root_agent = LlmAgent(
    name="basic_search_agent",
    model=adk_model(),                          # 【适配】gemini-2.0-flash-exp 已下线
    description="Agent to answer questions using Google Search.",
    instruction="I can answer your questions by searching the internet. Just ask me anything!",
    tools=[google_search],
)

runner = InMemoryRunner(root_agent)
session_id = str(uuid.uuid4())
# 【适配】create_session 是协程 → _sync 版；书里那句 asyncio.run 在 notebook 里会冲突
runner.session_service.create_session_sync(
    app_name=runner.app_name, user_id=USER_ID, session_id=session_id
)


def text_of(content) -> str:                    # 【适配】Content 没有 .text 属性
    if getattr(content, "text", None):
        return content.text
    return "".join(p.text for p in (getattr(content, "parts", None) or [])
                   if getattr(p, "text", None))


def collect():
    return list(runner.run(
        user_id=USER_ID,
        session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text="what's the latest ai news?")]),
    ))


try:
    for event in retry_call(collect, label="搜索 "):
        if event.is_final_response() and event.content:
            print("Agent Response:", text_of(event.content).strip())

            # 内建工具调没调，代码里看不到，只能靠 grounding 判断
            gm = getattr(event, "grounding_metadata", None)
            print("[grounding] 搜索词:", list(getattr(gm, "web_search_queries", None) or []) if gm else "(无)")
            print("[grounding] 引用来源:", len(getattr(gm, "grounding_chunks", None) or []) if gm else 0)
except Exception as e:
    print(f"❌ {type(e).__name__}: {short_error(e)}")


Agent Response: Major headlines across the artificial intelligence landscape focus on model evolutions, infrastructure scaling, corporate milestones, and workplace integration:

### 1. Frontier Models & Lab Updates
* **Anthropic Policy Updates & Public Market Moves:** Anthropic has filed for a public offering and announced upcoming revisions to its developer usage policies to address enterprise demand and responsible deployment standards.
* **Google's Ecosystem Expansions:** Google has introduced newer lightweight and expressive models within the Gemini family (such as Gemini Flash updates and expressive voice capabilities), alongside multi-modal creation features like Gemini Omni for video and avatar generation.
* **xAI Releases:** Updates to the Grok family (such as Grok 4.7) continue to target faster coding and specialized knowledge work, expanding tailored AI suites for government and high-compute enterprise users.

### 2. Semiconductor Growth & Infrastructure Scaling
* **Global Ch